# CivicStruct real-world evaluation candidates

This notebook creates a new 200-row evaluation candidate set and runs the five frozen CivicStruct systems on it. It does not train or change the model.

The rows come from two official public sources:

- 100 iChangeMyCity complaints, licensed CC BY-SA 2.0
- 100 City of Baton Rouge service requests, marked public domain

The source categories provide a useful service-domain diagnostic, but they are not full CivicStruct gold labels. The notebook therefore reports schema validity and mapped domain agreement only. It also writes a review CSV with a blank `gold_json` column for manual annotation. Full field metrics must wait for that review.

Before running, select **GPU T4 x2** and turn **Internet on** in Kaggle. Use **Run all** from a fresh session. The final ZIP is written to `/kaggle/working/civicstruct_real_world_evaluation.zip`.

In [ ]:
!pip -q install 'transformers==5.10.1' 'peft==0.20.0' 'bitsandbytes==0.50.0' 'accelerate==1.14.0' 'huggingface-hub==1.21.0' 'mlflow==3.15.1' 'scikit-learn==1.7.2'


In [ ]:
import csv
import hashlib
import html
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import mlflow
import torch
from huggingface_hub import snapshot_download

SEED = 42
PROJECT_REVISION = 'bf81ea53e6d781cf589fc84783c9e4e830b3504c'
RAW_BASE = f'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/{PROJECT_REVISION}'
MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
MODEL_REVISION = 'a07cc9a04f16550a088caea529712d1d335b0ac1'
SPACE_REPO = 'goyashek/civicstruct-grievance-demo'
SPACE_REVISION = '84fef0e6f66d3b09e4c46db2f21a0b2c88c5e8c9'
ADAPTER_SHA256 = '63934999afe7905ba1441f334b45ec31dd26e44fab5a03fa3c8ff82d611618a5'
MAX_NEW_TOKENS = 256
DEMO_COUNT = 3
STARTING_BATCH_SIZE = 8

WORK = Path('/kaggle/working')
ROOT = WORK / 'civicstruct_pinned_project'
MODEL_ROOT = WORK / 'civicstruct_models'
OUTPUT = WORK / 'civicstruct_real_world_output'
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True)
ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

gpu_count = torch.cuda.device_count()
assert torch.cuda.is_available() and gpu_count >= 1, 'Select a Kaggle GPU runtime before running this notebook.'
worker_count = min(2, gpu_count)
gpu_names = [torch.cuda.get_device_name(index) for index in range(gpu_count)]
if worker_count == 1:
    print('One GPU found. The notebook will still run, but T4 x2 is recommended.')
else:
    print('Two GPU workers will run in parallel:', gpu_names[:2])

def fetch_bytes(url, attempts=4, timeout=120):
    last_error = None
    for attempt in range(attempts):
        try:
            request = Request(url, headers={'User-Agent': 'CivicStruct-evaluation/1.0'})
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except (HTTPError, URLError, TimeoutError) as exc:
            last_error = exc
            if attempt + 1 < attempts:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'Could not fetch {url}. Check that Kaggle Internet is enabled.') from last_error

def fetch_json(url):
    return json.loads(fetch_bytes(url))

def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

PROJECT_FILES = (
    'src/schema.py',
    'data/dataset_manifest.json',
    'data/surface_variants.jsonl',
    'data/public_training_examples.jsonl',
    'data/test_cases.jsonl',
    'data/external_civic_eval.jsonl',
)
for relative_path in PROJECT_FILES:
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(fetch_bytes(f'{RAW_BASE}/{relative_path}'))

manifest = json.loads((ROOT / 'data/dataset_manifest.json').read_text(encoding='utf-8'))
for relative_path in (
    'data/surface_variants.jsonl',
    'data/public_training_examples.jsonl',
    'data/test_cases.jsonl',
    'data/external_civic_eval.jsonl',
):
    assert sha256_file(ROOT / relative_path) == manifest['sha256'][relative_path], relative_path

base_dir = Path(snapshot_download(
    repo_id=MODEL_NAME,
    revision=MODEL_REVISION,
    local_dir=MODEL_ROOT / 'base',
    allow_patterns=[
        '*.json', '*.jinja', '*.safetensors',
    ],
))
space_dir = Path(snapshot_download(
    repo_id=SPACE_REPO,
    repo_type='space',
    revision=SPACE_REVISION,
    local_dir=MODEL_ROOT / 'space',
    allow_patterns=['data/model_registry/artifacts/qlora_final_adapter/*'],
))
adapter_dir = space_dir / 'data/model_registry/artifacts/qlora_final_adapter'
adapter_weights = adapter_dir / 'adapter_model.safetensors'
assert adapter_weights.is_file(), adapter_weights
assert sha256_file(adapter_weights) == ADAPTER_SHA256, 'The downloaded adapter does not match the frozen registry record.'

sys.path.insert(0, str(ROOT))
from src.schema import validate_gold

print({
    'project_revision': PROJECT_REVISION,
    'dataset_version': manifest['dataset_version'],
    'model_revision': MODEL_REVISION,
    'adapter_verified': True,
    'gpu_count': gpu_count,
    'worker_count': worker_count,
})

## Curate the candidate rows

The source data is fetched at runtime. Exact addresses and locations are removed before screening. The screen then rejects digit-containing text, URLs, email addresses, phone-like strings, and common contact cues. Only the deidentified complaint, source category, and a row hash are kept.

This is a conservative automatic pass, not a privacy guarantee. The review sheet still needs a human privacy check before any candidate becomes a permanent project row.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

EMAIL_RE = re.compile(r'\b[^\s@]+@[^\s@]+\.[^\s@]+\b')
URL_RE = re.compile(r'\b(?:https?://|www\.)', re.IGNORECASE)
PHONE_RE = re.compile(r'(?:\+?\d[\s().-]*){10,}')
CONTACT_RE = re.compile(
    r'\b(?:my name|call me|contact me|email me|whatsapp|regards|mobile number|phone number)\b',
    re.IGNORECASE,
)

def squish(value):
    text = html.unescape(str(value or ''))
    text = re.sub(r'<[^>]+>', ' ', text)
    return ' '.join(text.split()).strip()

def normalized(text):
    return re.sub(r'\W+', ' ', text.casefold()).strip()

def redact_exact(text, phrases):
    clean_phrases = sorted({squish(value) for value in phrases if squish(value)}, key=len, reverse=True)
    for phrase in clean_phrases:
        text = re.sub(re.escape(phrase), ' ', text, flags=re.IGNORECASE)
    return squish(text)

def passes_text_screen(text):
    return (
        20 <= len(text) <= 500
        and not EMAIL_RE.search(text)
        and not URL_RE.search(text)
        and not PHONE_RE.search(text)
        and not CONTACT_RE.search(text)
        and not any(character.isdigit() for character in text)
        and '\ufffd' not in text
    )

def source_hash(row, fields):
    retained = {field: row.get(field) for field in fields}
    payload = json.dumps(retained, sort_keys=True, ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()

ICHANGE_API = 'https://data.opencity.in/api/3/action/datastore_search'
ICHANGE_RESOURCE_ID = 'a60abf5c-3a15-4967-af32-c3074248580f'
ICHANGE_CATEGORY_MAP = {
    'Public Transport - BMTC': 'public_transport',
    'Mobility - Roads, Public transport': 'public_transport',
    'Water Supply and Services': 'water_supply',
    'Garbage and Unsanitary Practices': 'sanitation_and_waste',
    'Yellow Spot': 'sanitation_and_waste',
    'Sewerage Systems': 'sanitation_and_waste',
    'Storm Water Drains': 'sanitation_and_waste',
    'Public Toilets': 'sanitation_and_waste',
    'Sanitation': 'sanitation_and_waste',
    'Mobility - Roads, Footpaths and Infrastructure': 'roads_and_streetlights',
    'Street lighting': 'roads_and_streetlights',
    'Streetlights': 'roads_and_streetlights',
    'Traffic and Road Safety': 'roads_and_streetlights',
    'Roads and Footpaths': 'roads_and_streetlights',
    'Electricity and Power Supply': 'electricity',
    'Certificates': 'welfare_or_document_service',
    'Animal Husbandry': 'other',
    'Pollution': 'other',
    'Others': 'other',
    'Parks & Recreation': 'other',
    'Trees and Saplings': 'other',
    'Community Infrastructure and Services': 'other',
    'Crime and Safety': 'other',
    'Lakes': 'other',
}
ICHANGE_QUOTAS = {
    'roads_and_streetlights': 24,
    'sanitation_and_waste': 20,
    'water_supply': 14,
    'electricity': 13,
    'public_transport': 8,
    'welfare_or_document_service': 6,
    'other': 15,
}

def make_ichange_text(row):
    title = squish(row.get('title'))
    description = squish(row.get('description'))
    if title and normalized(description).startswith(normalized(title)):
        body = description
    else:
        body = squish(f'{title}. {description}')
    body = redact_exact(body, (row.get('location'), row.get('address'))).strip(' .,:;-')
    if not passes_text_screen(body):
        return None
    ward = squish(row.get('ward_title'))
    if ward and passes_text_screen(f'placeholder text in {ward}'):
        body = f'In {ward}, {body[:1].lower()}{body[1:]}'
    complaint = body.rstrip('.') + '.'
    return complaint if passes_text_screen(complaint) else None

ichange_raw = []
offset = 0
while True:
    query = urlencode({'resource_id': ICHANGE_RESOURCE_ID, 'limit': 5000, 'offset': offset})
    page = fetch_json(f'{ICHANGE_API}?{query}')['result']['records']
    ichange_raw.extend(page)
    if len(page) < 5000:
        break
    offset += len(page)
assert len(ichange_raw) >= 16000, f'Unexpected iChangeMyCity row count: {len(ichange_raw)}'

ichange_pool = []
for source_row in ichange_raw:
    target_domain = ICHANGE_CATEGORY_MAP.get(source_row.get('category_title'))
    complaint = make_ichange_text(source_row)
    if target_domain is None or complaint is None:
        continue
    row_hash = source_hash(source_row, (
        'title', 'description', 'category_title', 'sub_category_title', 'ward_title',
    ))
    ichange_pool.append({
        'case_id': f'ichange-{row_hash[:16]}',
        'source_dataset': 'opencity_ichangemycity',
        'source_license': 'CC BY-SA 2.0',
        'source_row_sha256': row_hash,
        'source_category': squish(source_row.get('category_title')),
        'source_subcategory': squish(source_row.get('sub_category_title')),
        'target_domain': target_domain,
        'complaint': complaint,
    })

deduplicated = {}
for row in sorted(ichange_pool, key=lambda item: item['source_row_sha256']):
    deduplicated.setdefault(normalized(row['complaint']), row)
ichange_pool = list(deduplicated.values())

project_rows = []
for relative_path in (
    'data/surface_variants.jsonl',
    'data/public_training_examples.jsonl',
    'data/test_cases.jsonl',
    'data/external_civic_eval.jsonl',
):
    project_rows.extend(load_jsonl(ROOT / relative_path))
existing_complaints = [row['complaint'] for row in project_rows]
existing_hashes = {
    row['source_row_sha256']
    for row in project_rows
    if row.get('source_row_sha256')
}
ichange_pool = [row for row in ichange_pool if row['source_row_sha256'] not in existing_hashes]

vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
matrix = vectorizer.fit_transform(existing_complaints + [row['complaint'] for row in ichange_pool])
existing_matrix = matrix[:len(existing_complaints)]
candidate_matrix = matrix[len(existing_complaints):]
similarity = (existing_matrix @ candidate_matrix.T).tocsr()
near_overlap_indices = set()
for similarity_row in similarity:
    if similarity_row.nnz:
        order = similarity_row.data.argsort()[-25:]
        near_overlap_indices.update(int(index) for index in similarity_row.indices[order])
ichange_pool = [row for index, row in enumerate(ichange_pool) if index not in near_overlap_indices]

ichange_candidates = []
for domain, quota in ICHANGE_QUOTAS.items():
    domain_pool = sorted(
        (row for row in ichange_pool if row['target_domain'] == domain),
        key=lambda item: item['source_row_sha256'],
    )
    assert len(domain_pool) >= quota, {'domain': domain, 'available': len(domain_pool), 'needed': quota}
    ichange_candidates.extend(domain_pool[:quota])

BATON_API = 'https://data.brla.gov/resource/7ixm-mnvx.json'
BATON_PARENT_QUOTAS = {
    'GARBAGE': 12,
    'RECYCLING': 11,
    'DRAINAGE, EROSION, FLOODING OR HOLES': 11,
    'SEWER/WASTEWATER': 11,
    'ROAD MAINTENANCE ISSUES': 18,
    'STREET/TRAFFIC ISSUES': 17,
    'BLIGHTED PROPERTIES': 4,
    'BUILDING CODE/ZONING VIOLATIONS': 4,
    'ENVIRONMENTAL ISSUES': 4,
    'NEIGHBORHOOD/SUBDIVISION ISSUES': 4,
    'MOWING AND TREE ISSUES': 4,
}
BATON_CATEGORY_MAP = {
    parent: (
        'sanitation_and_waste'
        if parent in {'GARBAGE', 'RECYCLING', 'DRAINAGE, EROSION, FLOODING OR HOLES', 'SEWER/WASTEWATER'}
        else 'roads_and_streetlights'
        if parent in {'ROAD MAINTENANCE ISSUES', 'STREET/TRAFFIC ISSUES'}
        else 'other'
    )
    for parent in BATON_PARENT_QUOTAS
}

def make_baton_text(row):
    body = redact_exact(squish(row.get('comments')), (row.get('streetaddress'),))
    body = re.sub(
        r'\b(?:caller|resident|customer)\s+(?:states?|reports?)\b\s*',
        '', body, flags=re.IGNORECASE,
    ).strip(' .,:;-')
    if not passes_text_screen(body):
        return None
    complaint = f'In Baton Rouge, {body[:1].lower()}{body[1:].rstrip(".")}.'
    return complaint if passes_text_screen(complaint) else None

baton_candidates = []
baton_pool_counts = {}
for parent_type, quota in BATON_PARENT_QUOTAS.items():
    escaped_parent = parent_type.replace("'", "''")
    where = (
        "createdate between '2024-01-01T00:00:00.000' and '2024-12-31T23:59:59.999' "
        f"AND comments is not null AND parenttype = '{escaped_parent}'"
    )
    query = urlencode({
        '$select': 'parenttype,typename,comments,streetaddress,cityname,createdate',
        '$where': where,
        '$order': 'id ASC',
        '$limit': 1500,
    })
    source_rows = fetch_json(f'{BATON_API}?{query}')
    parent_pool = []
    for source_row in source_rows:
        complaint = make_baton_text(source_row)
        if complaint is None:
            continue
        row_hash = source_hash(source_row, (
            'parenttype', 'typename', 'comments', 'cityname', 'createdate',
        ))
        parent_pool.append({
            'case_id': f'baton-{row_hash[:16]}',
            'source_dataset': 'baton_rouge_311',
            'source_license': 'Public domain',
            'source_row_sha256': row_hash,
            'source_category': parent_type,
            'source_subcategory': squish(source_row.get('typename')),
            'target_domain': BATON_CATEGORY_MAP[parent_type],
            'complaint': complaint,
        })
    unique_parent_pool = {}
    for row in sorted(parent_pool, key=lambda item: item['source_row_sha256']):
        unique_parent_pool.setdefault(normalized(row['complaint']), row)
    parent_pool = list(unique_parent_pool.values())
    baton_pool_counts[parent_type] = len(parent_pool)
    assert len(parent_pool) >= quota, {'parent_type': parent_type, 'available': len(parent_pool), 'needed': quota}
    baton_candidates.extend(parent_pool[:quota])

candidates = sorted(ichange_candidates, key=lambda row: row['source_row_sha256'])
candidates += sorted(baton_candidates, key=lambda row: row['source_row_sha256'])
assert len(ichange_candidates) == 100
assert len(baton_candidates) == 100
assert len(candidates) == 200
assert len({row['case_id'] for row in candidates}) == 200
assert len({normalized(row['complaint']) for row in candidates}) == 200
assert all(passes_text_screen(row['complaint']) for row in candidates)

candidate_path = OUTPUT / 'candidate_rows.jsonl'
candidate_path.write_text(
    ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in candidates),
    encoding='utf-8',
)
source_metadata = {
    'candidate_set_role': 'manual-review candidates for Evaluation v3',
    'full_civicstruct_gold_available': False,
    'source_category_is_weak_label': True,
    'privacy_review_still_required': True,
    'ichange_my_city': {
        'landing_page': 'https://data.opencity.in/dataset/ichangemycity-complaints',
        'api_resource_id': ICHANGE_RESOURCE_ID,
        'license': 'CC BY-SA 2.0',
        'raw_rows_seen': len(ichange_raw),
        'safe_rows_before_overlap_screen': len(deduplicated),
        'near_overlap_rows_removed': len(near_overlap_indices),
        'selected_rows': len(ichange_candidates),
        'domain_quotas': ICHANGE_QUOTAS,
    },
    'baton_rouge': {
        'landing_page': 'https://data.brla.gov/d/7ixm-mnvx',
        'license': 'Public domain',
        'fixed_date_window': '2024-01-01 through 2024-12-31',
        'safe_pool_counts': baton_pool_counts,
        'selected_rows': len(baton_candidates),
        'parent_type_quotas': BATON_PARENT_QUOTAS,
        'mapped_domains': BATON_CATEGORY_MAP,
    },
    'retained_candidate_fields': list(candidates[0]),
    'dropped_source_fields': ['raw identifiers', 'exact addresses', 'coordinates', 'contact details'],
    'automatic_text_screen': '20 to 500 characters; no digits, URLs, emails, phone-like text, contact cues, or replacement characters',
}
save_json(OUTPUT / 'source_metadata.json', source_metadata)
print({
    'candidate_rows': len(candidates),
    'sources': Counter(row['source_dataset'] for row in candidates),
    'mapped_domains': Counter(row['target_domain'] for row in candidates),
    'near_overlap_rows_removed': len(near_overlap_indices),
})

## Freeze the prompts and split work across the GPUs

Retrieval uses only the frozen 160-row training set. The two worker processes receive alternating candidate rows. Each process can see only one physical GPU, so both T4s load one 4-bit base model and generate independently.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

controlled_rows = [
    row for row in load_jsonl(ROOT / 'data/surface_variants.jsonl')
    if row['split'] == 'train'
]
public_training = load_jsonl(ROOT / 'data/public_training_examples.jsonl')
training = controlled_rows + public_training
assert len(controlled_rows) == 120
assert len(public_training) == 40
assert len(training) == 160
assert len({row['case_id'] for row in training}) == 160

STATIC_IDS = ('canonical-001', 'canonical-020', 'canonical-030')
training_by_id = {row['case_id']: row for row in training}
assert all(case_id in training_by_id for case_id in STATIC_IDS)

retrieval_vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
training_matrix = retrieval_vectorizer.fit_transform([row['complaint'] for row in training])
candidate_matrix = retrieval_vectorizer.transform([row['complaint'] for row in candidates])
similarities = candidate_matrix @ training_matrix.T
for row, scores in zip(candidates, similarities):
    ranked = scores.toarray().ravel().argsort()[::-1][:DEMO_COUNT]
    row['retrieved_demo_ids'] = [training[index]['case_id'] for index in ranked]
    assert len(set(row['retrieved_demo_ids'])) == DEMO_COUNT

DOMAINS = ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']
ISSUES = ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']
URGENCY = ['routine', 'time_sensitive', 'safety_critical']
MISSING = ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']
SYSTEM_PROMPT = (
    'Structure one public-service complaint as exactly one JSON object. Use these fields in this order: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, urgency, missing_information, formal_summary. '
    f'Allowed service_domain values: {DOMAINS}. Allowed issue_type values: {ISSUES}. '
    f'Allowed urgency values: {URGENCY}. Allowed missing_information values: {MISSING}. '
    'Use null for absent scalar facts. Missing information must be a non-empty ordered list. Use the none label only when no important detail is missing. '
    'Do not guess facts. The formal summary must be one neutral sentence. Return no reasoning, markdown, or commentary.'
)

def rule_output(complaint):
    text = complaint.casefold()
    if any(word in text for word in ('water', 'tap ', 'tanker')):
        domain = 'water_supply'
    elif any(word in text for word in ('garbage', 'waste', 'sewage', 'sanitation', 'bin ', 'sweeper')):
        domain = 'sanitation_and_waste'
    elif any(word in text for word in ('electric', 'power', 'feeder', 'transformer')):
        domain = 'electricity'
    elif any(word in text for word in ('road', 'streetlight', 'street light', 'pothole', 'signal', 'sidewalk', 'pavement', 'parking meter')):
        domain = 'roads_and_streetlights'
    elif any(word in text for word in ('pension', 'certificate', 'benefit', 'welfare', 'application', 'document')):
        domain = 'welfare_or_document_service'
    elif any(word in text for word in ('bus', 'train', 'station', 'platform', 'ticket', 'route ')):
        domain = 'public_transport'
    else:
        domain = 'other'
    if any(word in text for word in ('charged', 'bill ', 'fee ', 'payment')):
        issue = 'overcharging_or_payment_problem'
    elif any(word in text for word in ('wrong', 'incorrect', 'marked completed', 'record ')):
        issue = 'record_or_document_error'
    elif any(word in text for word in ('shouted', 'insulted', 'rude', 'refused', 'mocked', 'ignored')):
        issue = 'staff_conduct'
    elif any(word in text for word in ('live wire', 'sparking', 'unsafe', 'hazard', 'collision', 'needles', 'sharp edges')):
        issue = 'safety_or_health_hazard'
    elif any(word in text for word in ('did not arrive', 'never arrived', 'never came', 'did not come', 'scheduled for', 'was due')):
        issue = 'delay_or_non_arrival'
    elif any(word in text for word in ('broken', 'cracked', 'damaged', 'pothole', 'leaking', 'raised sidewalk', 'leaning')):
        issue = 'damaged_infrastructure'
    elif any(word in text for word in ('no water', 'no power', 'unavailable', 'not working', 'has not worked', 'error for', 'no collection')):
        issue = 'service_outage_or_non_delivery'
    else:
        issue = 'other'
    urgency = (
        'safety_critical' if issue == 'safety_or_health_hazard'
        else 'time_sensitive' if any(word in text for word in (
            'for two days', 'for three days', 'for four days', 'for five days',
            'for six days', 'for a week', 'since monday', 'since friday', 'since sunday',
        ))
        else 'routine'
    )
    return {
        'service_domain': domain,
        'issue_type': issue,
        'location': None,
        'event_date_or_time': None,
        'amount_inr': None,
        'service_identifier': None,
        'urgency': urgency,
        'missing_information': ['exact_location'],
        'formal_summary': complaint.strip(),
    }

rules_outputs = []
for row in candidates:
    rules_outputs.append({
        'case_id': row['case_id'],
        'response': json.dumps(rule_output(row['complaint']), ensure_ascii=False, separators=(',', ':')),
        'latency_seconds': 0.0,
        'prompt_tokens': 0,
        'worker_index': None,
        'batch_size_used': None,
    })

payload = {
    'base_dir': str(base_dir),
    'adapter_dir': str(adapter_dir),
    'project_root': str(ROOT),
    'output_dir': str(OUTPUT),
    'system_prompt': SYSTEM_PROMPT,
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'max_new_tokens': MAX_NEW_TOKENS,
    'starting_batch_size': STARTING_BATCH_SIZE,
    'static_ids': list(STATIC_IDS),
    'training': training,
    'candidates': candidates,
}
payload_path = WORK / 'civicstruct_gpu_payload.json'
save_json(payload_path, payload)
print({
    'training_rows': len(training),
    'candidate_rows': len(candidates),
    'static_demo_ids': list(STATIC_IDS),
    'retrieval': 'TF-IDF over frozen training rows only',
})

In [ ]:
WORKER_SOURCE = r"""import gc
import json
import sys
import time
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

payload_path = Path(sys.argv[1])
worker_index = int(sys.argv[2])
worker_count = int(sys.argv[3])
payload = json.loads(payload_path.read_text(encoding='utf-8'))
sys.path.insert(0, payload['project_root'])

training_by_id = {row['case_id']: row for row in payload['training']}
rows = payload['candidates'][worker_index::worker_count]
base_dir = payload['base_dir']
adapter_dir = payload['adapter_dir']
starting_batch_size = payload['starting_batch_size']
max_new_tokens = payload['max_new_tokens']
system_prompt = payload['system_prompt']

assert torch.cuda.is_available(), 'The worker cannot see its assigned GPU.'
assert torch.cuda.device_count() == 1, f'Worker {worker_index} can see {torch.cuda.device_count()} GPUs instead of one.'
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

def messages_for(complaint, demos=()):
    messages = [{'role': 'system', 'content': system_prompt}]
    for demo in demos:
        messages.extend([
            {'role': 'user', 'content': demo['complaint']},
            {'role': 'assistant', 'content': json.dumps(demo['gold'], ensure_ascii=False, separators=(',', ':'))},
        ])
    messages.append({'role': 'user', 'content': complaint})
    return messages

def encoded_batch(tokenizer, message_batch):
    kwargs = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_dict': True,
        'return_tensors': 'pt',
        'padding': True,
        'truncation': True,
        'max_length': 2048,
        'enable_thinking': False,
    }
    try:
        return tokenizer.apply_chat_template(message_batch, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking')
        return tokenizer.apply_chat_template(message_batch, **kwargs)

def demos_for(row, method):
    if method == 'static_few_shot':
        return [training_by_id[case_id] for case_id in payload['static_ids']]
    if method == 'retrieved_few_shot':
        return [training_by_id[case_id] for case_id in row['retrieved_demo_ids']]
    return []

def generate_rows(tokenizer, model, method):
    outputs = []
    position = 0
    batch_size = starting_batch_size
    model.eval()
    while position < len(rows):
        batch = rows[position:position + batch_size]
        message_batch = [messages_for(row['complaint'], demos_for(row, method)) for row in batch]
        try:
            inputs = encoded_batch(tokenizer, message_batch).to('cuda:0')
            prompt_lengths = inputs['attention_mask'].sum(dim=1).tolist()
            padded_length = inputs['input_ids'].shape[1]
            torch.cuda.synchronize()
            started = time.perf_counter()
            with torch.inference_mode():
                generated = model.generate(
                    **inputs,
                    do_sample=False,
                    max_new_tokens=max_new_tokens,
                    use_cache=True,
                    pad_token_id=tokenizer.pad_token_id,
                )
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            responses = tokenizer.batch_decode(generated[:, padded_length:], skip_special_tokens=True)
            for row, response, prompt_tokens in zip(batch, responses, prompt_lengths):
                outputs.append({
                    'case_id': row['case_id'],
                    'response': response.strip(),
                    'latency_seconds': elapsed / len(batch),
                    'prompt_tokens': int(prompt_tokens),
                    'worker_index': worker_index,
                    'batch_size_used': len(batch),
                })
            position += len(batch)
            del inputs, generated
            print(json.dumps({'worker': worker_index, 'method': method, 'completed': position, 'rows': len(rows), 'batch_size': batch_size}), flush=True)
        except torch.OutOfMemoryError:
            if 'inputs' in locals():
                del inputs
            if 'generated' in locals():
                del generated
            gc.collect()
            torch.cuda.empty_cache()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print(json.dumps({'worker': worker_index, 'method': method, 'oom_backoff_batch_size': batch_size}), flush=True)
    return outputs

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(base_dir, local_files_only=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    base_dir,
    local_files_only=True,
    quantization_config=quantization,
    dtype=torch.float16,
    device_map={'': 0},
)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.config.use_cache = True

results = {}
for method in ('zero_shot', 'static_few_shot', 'retrieved_few_shot'):
    results[method] = generate_rows(tokenizer, model, method)
    gc.collect()
    torch.cuda.empty_cache()

model = PeftModel.from_pretrained(model, adapter_dir, is_trainable=False)
results['qlora'] = generate_rows(tokenizer, model, 'qlora')
result = {
    'worker_index': worker_index,
    'visible_gpu': torch.cuda.get_device_name(0),
    'rows': len(rows),
    'methods': results,
}
output_path = Path(payload['output_dir']) / f'gpu_worker_{worker_index}.json'
output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps({'worker': worker_index, 'finished': True, 'output': str(output_path)}), flush=True)
"""

worker_path = WORK / '_civicstruct_gpu_worker.py'
worker_path.write_text(WORKER_SOURCE, encoding='utf-8')
processes = []
log_handles = []
inference_started = time.perf_counter()
for worker_index in range(worker_count):
    log_path = OUTPUT / f'gpu_worker_{worker_index}.log'
    log_handle = log_path.open('w', encoding='utf-8')
    environment = os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = str(worker_index)
    environment['TOKENIZERS_PARALLELISM'] = 'false'
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(
        [sys.executable, str(worker_path), str(payload_path), str(worker_index), str(worker_count)],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=environment,
    )
    processes.append(process)
    log_handles.append(log_handle)

last_update = 0.0
while any(process.poll() is None for process in processes):
    now = time.monotonic()
    if now - last_update >= 30:
        print({'worker_status': [process.poll() for process in processes], 'elapsed_minutes': round((time.perf_counter() - inference_started) / 60, 1)})
        last_update = now
    time.sleep(5)

for log_handle in log_handles:
    log_handle.close()
return_codes = [process.returncode for process in processes]
if any(code != 0 for code in return_codes):
    log_tails = {}
    for worker_index in range(worker_count):
        lines = (OUTPUT / f'gpu_worker_{worker_index}.log').read_text(encoding='utf-8').splitlines()
        log_tails[f'worker_{worker_index}'] = '\n'.join(lines[-40:])
    raise RuntimeError(f'GPU worker failure: {return_codes}\n{json.dumps(log_tails, indent=2)}')

inference_wall_seconds = time.perf_counter() - inference_started
print({'workers_finished': worker_count, 'return_codes': return_codes, 'wall_minutes': round(inference_wall_seconds / 60, 1)})

## Merge and check every prediction

Every system must return exactly one response per candidate. Strict validity uses the frozen schema validator. Domain agreement compares the predicted `service_domain` with the mapped public source category and is reported separately for each source.

In [ ]:
method_outputs = {'deterministic_rules': rules_outputs}
worker_records = [
    json.loads((OUTPUT / f'gpu_worker_{index}.json').read_text(encoding='utf-8'))
    for index in range(worker_count)
]
for method in ('zero_shot', 'static_few_shot', 'retrieved_few_shot', 'qlora'):
    combined = []
    for worker_record in worker_records:
        combined.extend(worker_record['methods'][method])
    method_outputs[method] = combined

candidate_ids = [row['case_id'] for row in candidates]
candidate_by_id = {row['case_id']: row for row in candidates}
for method, outputs in method_outputs.items():
    output_ids = [item['case_id'] for item in outputs]
    assert len(outputs) == len(candidates), {'method': method, 'outputs': len(outputs)}
    assert len(set(output_ids)) == len(candidates), {'method': method, 'duplicate_ids': True}
    assert set(output_ids) == set(candidate_ids), {'method': method, 'missing_or_extra_ids': True}
    outputs.sort(key=lambda item: candidate_ids.index(item['case_id']))

def parse_strict(raw):
    try:
        parsed = json.loads(raw)
        validate_gold(parsed)
        return parsed
    except (json.JSONDecodeError, TypeError, ValueError):
        return None

def diagnostic_for(outputs):
    rows = []
    for output in outputs:
        candidate = candidate_by_id[output['case_id']]
        parsed = parse_strict(output['response'])
        rows.append({
            **output,
            'source_dataset': candidate['source_dataset'],
            'source_category': candidate['source_category'],
            'target_domain': candidate['target_domain'],
            'strict_schema_valid': parsed is not None,
            'predicted_service_domain': None if parsed is None else parsed['service_domain'],
            'source_domain_match': parsed is not None and parsed['service_domain'] == candidate['target_domain'],
        })
    valid_rows = [row for row in rows if row['strict_schema_valid']]
    result = {
        'total_rows': len(rows),
        'strict_schema_valid_count': len(valid_rows),
        'strict_schema_validity_rate': len(valid_rows) / len(rows),
        'domain_match_rate_end_to_end': sum(row['source_domain_match'] for row in rows) / len(rows),
        'domain_match_rate_valid_only': (
            None if not valid_rows
            else sum(row['source_domain_match'] for row in valid_rows) / len(valid_rows)
        ),
        'mean_latency_seconds': sum(row['latency_seconds'] for row in rows) / len(rows),
        'mean_prompt_tokens': sum(row['prompt_tokens'] for row in rows) / len(rows),
        'by_source': {},
    }
    for source in sorted({row['source_dataset'] for row in rows}):
        source_rows = [row for row in rows if row['source_dataset'] == source]
        source_valid = [row for row in source_rows if row['strict_schema_valid']]
        result['by_source'][source] = {
            'rows': len(source_rows),
            'strict_schema_validity_rate': len(source_valid) / len(source_rows),
            'domain_match_rate_end_to_end': sum(row['source_domain_match'] for row in source_rows) / len(source_rows),
            'domain_match_rate_valid_only': (
                None if not source_valid
                else sum(row['source_domain_match'] for row in source_valid) / len(source_valid)
            ),
        }
    return result, rows

diagnostics = {}
detailed_outputs = {}
for method, outputs in method_outputs.items():
    diagnostics[method], detailed_outputs[method] = diagnostic_for(outputs)
    save_json(OUTPUT / f'{method}_predictions.json', {
        'method': method,
        'model_name': None if method == 'deterministic_rules' else MODEL_NAME,
        'model_revision': None if method == 'deterministic_rules' else MODEL_REVISION,
        'adapter_sha256': ADAPTER_SHA256 if method == 'qlora' else None,
        'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
        'diagnostic': diagnostics[method],
        'outputs': detailed_outputs[method],
    })

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri((OUTPUT / 'mlruns').as_uri())
mlflow.set_experiment('civicstruct-real-world-evaluation-v3-candidates')
mlflow_run_ids = {}
for method, diagnostic in diagnostics.items():
    with mlflow.start_run(run_name=method) as run:
        mlflow_run_ids[method] = run.info.run_id
        mlflow.log_params({
            'method': method,
            'rows': len(candidates),
            'candidate_role': 'manual_review_candidates',
            'full_gold_available': False,
            'model_name': 'none' if method == 'deterministic_rules' else MODEL_NAME,
            'model_revision': 'none' if method == 'deterministic_rules' else MODEL_REVISION,
            'worker_count': worker_count,
        })
        mlflow.log_metrics({
            'strict_schema_validity_rate': diagnostic['strict_schema_validity_rate'],
            'domain_match_rate_end_to_end': diagnostic['domain_match_rate_end_to_end'],
            'mean_latency_seconds': diagnostic['mean_latency_seconds'],
            'mean_prompt_tokens': diagnostic['mean_prompt_tokens'],
        })
        mlflow.log_artifact(str(OUTPUT / f'{method}_predictions.json'), artifact_path='predictions')

print(json.dumps({
    method: {
        'schema_validity': round(record['strict_schema_validity_rate'], 4),
        'mapped_domain_agreement': round(record['domain_match_rate_end_to_end'], 4),
    }
    for method, record in diagnostics.items()
}, indent=2))

In [ ]:
response_lookup = {
    method: {item['case_id']: item['response'] for item in outputs}
    for method, outputs in method_outputs.items()
}
review_path = OUTPUT / 'review_sheet.csv'
with review_path.open('w', encoding='utf-8', newline='') as handle:
    fieldnames = [
        'case_id', 'source_dataset', 'source_license', 'source_category',
        'source_subcategory', 'mapped_target_domain', 'complaint',
        'deterministic_rules_raw', 'zero_shot_raw', 'static_few_shot_raw',
        'retrieved_few_shot_raw', 'qlora_raw', 'privacy_checked',
        'review_status', 'gold_json', 'reviewer_notes',
    ]
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for row in candidates:
        writer.writerow({
            'case_id': row['case_id'],
            'source_dataset': row['source_dataset'],
            'source_license': row['source_license'],
            'source_category': row['source_category'],
            'source_subcategory': row['source_subcategory'],
            'mapped_target_domain': row['target_domain'],
            'complaint': row['complaint'],
            'deterministic_rules_raw': response_lookup['deterministic_rules'][row['case_id']],
            'zero_shot_raw': response_lookup['zero_shot'][row['case_id']],
            'static_few_shot_raw': response_lookup['static_few_shot'][row['case_id']],
            'retrieved_few_shot_raw': response_lookup['retrieved_few_shot'][row['case_id']],
            'qlora_raw': response_lookup['qlora'][row['case_id']],
            'privacy_checked': '',
            'review_status': '',
            'gold_json': '',
            'reviewer_notes': '',
        })

review_notes = """# Manual review needed

These 200 rows are candidates, not a finished gold test set.

For each row:

1. Check the complaint again for names, exact addresses, contact details, or other personal information. Reject or safely rewrite any questionable row.
2. Confirm that the mapped source category is reasonable. It is a weak diagnostic label, not automatic CivicStruct gold.
3. Fill `gold_json` with all nine CivicStruct fields and set `review_status` to `accepted`, `edited`, or `rejected`.
4. Keep rejected rows out of later metrics.
5. Run a separate scoring pass only after every accepted row has a reviewed `gold_json` value.

This notebook does not train on these rows and does not change Evaluation v2.
"""
(OUTPUT / 'REVIEW_INSTRUCTIONS.md').write_text(review_notes, encoding='utf-8')

environment = {
    'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'torch': torch.__version__,
    'cuda_runtime': torch.version.cuda,
    'gpu_count': gpu_count,
    'gpu_names': gpu_names,
    'worker_count_used': worker_count,
    'packages': {
        package: version(package)
        for package in (
            'transformers', 'peft', 'bitsandbytes', 'accelerate',
            'huggingface-hub', 'mlflow', 'scikit-learn',
        )
    },
    'project_revision': PROJECT_REVISION,
    'dataset_version': manifest['dataset_version'],
    'base_model': MODEL_NAME,
    'base_model_revision': MODEL_REVISION,
    'adapter_space_revision': SPACE_REVISION,
    'adapter_sha256': ADAPTER_SHA256,
    'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
    'starting_batch_size': STARTING_BATCH_SIZE,
    'inference_wall_seconds': inference_wall_seconds,
}
try:
    environment['nvidia_smi'] = subprocess.check_output(
        ['nvidia-smi'], text=True, timeout=30,
    )
except (FileNotFoundError, subprocess.SubprocessError):
    environment['nvidia_smi'] = None
save_json(OUTPUT / 'environment.json', environment)

run_summary = {
    'candidate_rows': len(candidates),
    'source_counts': dict(Counter(row['source_dataset'] for row in candidates)),
    'mapped_domain_counts': dict(Counter(row['target_domain'] for row in candidates)),
    'systems': diagnostics,
    'mlflow_run_ids': mlflow_run_ids,
    'worker_count': worker_count,
    'inference_wall_seconds': inference_wall_seconds,
    'full_civicstruct_gold_available': False,
    'full_field_metrics_computed': False,
    'evaluation_v2_changed': False,
    'next_step': 'manually review privacy and fill gold_json in review_sheet.csv',
}
save_json(OUTPUT / 'run_summary.json', run_summary)

assert review_path.is_file() and review_path.stat().st_size > 0
assert all((OUTPUT / f'{method}_predictions.json').is_file() for method in method_outputs)
assert (OUTPUT / 'source_metadata.json').is_file()
assert (OUTPUT / 'environment.json').is_file()
assert len(list(csv.DictReader(review_path.open(encoding='utf-8')))) == 200

worker_path.unlink(missing_ok=True)
payload_path.unlink(missing_ok=True)
archive_path = Path(shutil.make_archive(
    str(WORK / 'civicstruct_real_world_evaluation'),
    'zip',
    root_dir=str(OUTPUT),
))
assert archive_path.is_file() and archive_path.stat().st_size > 0
archive_listing = subprocess.check_output(['unzip', '-t', str(archive_path)], text=True)
assert 'No errors detected' in archive_listing
print({
    'archive': str(archive_path),
    'archive_mb': round(archive_path.stat().st_size / 1_000_000, 2),
    'review_rows': 200,
    'full_field_metrics_computed': False,
    'next_step': 'review review_sheet.csv and fill gold_json',
})

## What this run proves

A successful run proves that the frozen systems can process the new real-world candidate set, and it records strict JSON validity plus agreement with coarse public source categories. It does not prove full extraction quality because location, time, issue type, urgency, missing information, and factuality still need human gold labels.

Keep `civicstruct_real_world_evaluation.zip` as the untouched run artifact. Annotate a copy of `review_sheet.csv`, then build a separate scoring notebook after the review is complete.